# Running PESTPP-IES

In [ ]:
import os
import shutil
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning) 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt;

import pyemu
import herebedragons as hbd

In [ ]:
# specify the temporary working folder
t_d = os.path.join('master_ies')
# get the weighted PEST dataset + prior ensemble produced by the prior-MC / truth notebook
org_t_d = os.path.join("priormc")
if not os.path.exists(org_t_d):
    raise Exception()
if os.path.exists(t_d):
    shutil.rmtree(t_d)
shutil.copytree(org_t_d,t_d)

In [ ]:
pst_path = os.path.join(t_d, 'pest.pst')
pst = pyemu.Pst(pst_path)

In [ ]:
# check to see if obs&weights notebook has been run
if not pst.nnz_obs > 0:
    raise Exception()

Here we set some handy PESTPP-IES options:

- `ies_bad_phi_sigma`: a realization is flagged "bad" if its objective function is much worse than the ensemble average, with the cutoff set by the standard deviation scaled by this number.
- `ies_multimodal_alpha`: the fraction of the total ensemble used as the local neighbourhood in the multimodal solution process.
- `ies_n_iter_reinflate`: the number of between-iteration covariance re-inflations.

In [ ]:
pst.pestpp_options["ies_num_reals"] = 100
pst.pestpp_options["ies_bad_phi_sigma"] = 1.75
#pst.pestpp_options["ies_multimodal_alpha"] = 0.99

## Observation noise



In [ ]:
obs = pst.observation_data
obs["standard_deviation"] = np.nan
nzobs = obs.loc[obs.weight > 0, :].copy()

# heads (m) and GDE pH (pH units): sigma = 0.1
obs.loc[nzobs.obsnme, "standard_deviation"] = 0.1

# GDE drain flux: sigma = 20% of |value|  (value is negative -> use abs)
dobs = nzobs.loc[nzobs.usecol == "drn-gde", :]
assert dobs.shape[0] == 1
obs.loc[dobs.obsnme, "standard_deviation"] = np.abs(dobs.obsval.values) * 0.2

# build a (correlated) observation-noise ensemble: one common standard-normal
# factor per realization, scaled by each observation's standard deviation
nzobs = obs.loc[obs.weight > 0, :].copy()
np.random.seed(pyemu.en.SEED)
draws = np.random.normal(0.0, 1.0, pst.pestpp_options["ies_num_reals"])
vals = nzobs.obsval.values
stdevs = nzobs.standard_deviation.values
noisyobs = [vals + d * stdevs for d in draws]
df = pd.DataFrame(noisyobs,
                  index=np.arange(pst.pestpp_options["ies_num_reals"]),
                  columns=nzobs.obsnme)
df.to_csv(os.path.join(t_d, "corrnoise.csv"))
pst.pestpp_options["ies_obs_en"] = "corrnoise.csv"

Let's run PESTPP-IES with `noptmax = -2`. This computes the mean of the initial parameter ensemble, evaluates it (one model run), and records the results.

In [ ]:
pst.control_data.noptmax = -2 
pst.write(os.path.join(t_d, 'pest.pst'),version=2)

pyemu.os_utils.run("pestpp-ies pest.pst",cwd=t_d)

## More weighting adjustments

The `ies_phi_factor_file` option turns on internal weight adjustment. It points to a file with two columns and no header (comma- or space-delimited):

- Column 1: a tag matched against observation-group names by substring (lets you combine groups into one "weighting group").
- Column 2: a positive number (phi factor) -- the share of the current measurement phi that group should take.

Here we have three observation groups -- heads (`hds`), the GDE drain flux (`drn`), and the GDE pH (`ph`) -- and we split the objective-function contribution roughly evenly across them.

In [ ]:
with open(os.path.join(t_d, "phi.csv"), "w") as f:
    f.write("hds,0.34\n")   # heads
    f.write("drn,0.33\n")   # GDE drain flux
    f.write("ph,0.33\n")    # GDE pH (management quantity)
# pst.pestpp_options["ies_phi_factor_file"] = "phi.csv"

In [ ]:
pst.control_data.noptmax = -2
pst.write(os.path.join(t_d, 'pest.pst'),version=2)

pyemu.os_utils.run("pestpp-ies pest.pst",cwd=t_d)

## For the Win!

In [ ]:
pst.control_data.noptmax = 3
pst.write(os.path.join(t_d, 'pest.pst'),version=2)

In [ ]:
num_workers = 20
m_d = "master_ies0"

In [ ]:
pyemu.os_utils.start_workers(t_d, # the folder which contains the "template" PEST dataset
                            'pestpp-ies', #the PEST software version we want to run
                            'pest.pst', # the control file to use with PEST
                            num_workers=num_workers, #how many agents to deploy
                            worker_root='.', #where to deploy the agent directories; relative to where python is running
                            master_dir=m_d, #the manager directory
                            )

In [ ]:
pst = pyemu.Pst(os.path.join(m_d, "pest.pst"))
obs = pst.observation_data
oe_pr = pst.ies.obsen0
oe_pt = pst.ies.get("obsen", pst.ies.phiactual.iteration.max())
forecasts = [f.strip() for f in pst.pestpp_options["forecasts"].split(",")]

fig, axs = plt.subplots(len(forecasts), 1, figsize=(6, 2.3 * len(forecasts)),
                        constrained_layout=True)
for f, ax in zip(forecasts, np.atleast_1d(axs)):
    oe_pr.loc[:, f].plot(kind="hist", fc="0.5", alpha=0.5, density=True, ax=ax, label="prior")
    oe_pt.loc[:, f].plot(kind="hist", fc="b", alpha=0.5, density=True, ax=ax, label="posterior")
    ylim = ax.get_ylim()
    v = obs.loc[f, "obsval"]
    ax.plot([v, v], ylim, "r-", lw=2, label="truth")
    ax.set_title(" ".join(f.split("usecol:")[-1].split("_")), fontsize=9)
    ax.set_yticks([]); ax.set_ylabel("")
np.atleast_1d(axs)[0].legend()

## Parameter field recovery: prior vs posterior vs truth

For each spatially varying field -- hydraulic conductivity, porosity, and calcite -- compare
the prior, the posterior (after assimilation), and the truth. Tighter posterior-to-truth
agreement means the data informed that field.

In [ ]:
tpst = pyemu.Pst(os.path.join("master_priormc", "truth.pst"))

In [ ]:
# helper functions to map a field observation set (hk, poro, calcite) to a 2-D array
# and compare prior vs posterior vs truth
def field_obs(oname):
    f = obs.loc[obs.oname == oname, :].copy()
    f["i"] = f.i.astype(int)
    f["j"] = f.j.astype(int)
    return f

def field_to_arr(fobs, series):
    arr = np.full((fobs.i.max() + 1, fobs.j.max() + 1), np.nan)
    arr[fobs.i, fobs.j] = series.loc[fobs.obsnme].values
    return arr

def plot_field(oname, real="base", log=False, title=None):
    fobs = field_obs(oname)
    pr = field_to_arr(fobs, oe_pr.loc[real, :])
    pt = field_to_arr(fobs, oe_pt.loc[real, :])
    tr = field_to_arr(fobs, tpst.res.loc[:, "modelled"])
    if log:
        pr, pt, tr = np.log10(pr), np.log10(pt), np.log10(tr)
    vmin = np.nanmin([pr, pt, tr])
    vmax = np.nanmax([pr, pt, tr])
    lab = title or oname
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
    for ax, arr, sub in zip(axes, [pr, pt, tr], ["Prior", "Posterior", "Truth"]):
        cb = ax.imshow(arr, vmin=vmin, vmax=vmax)
        ttl = f"{sub} {lab}" if sub == "Truth" else f"{sub} {lab} (real {real})"
        ax.set_title(ttl)
        plt.colorbar(cb, ax=ax, fraction=0.046, pad=0.04)
    plt.show()

In [ ]:
# parameter field recovery for the base realization
plot_field("hk", real="base", log=True, title="H$_k$ (m/d)")
plot_field("poro", real="base", title="Porosity")
plot_field("calcite", real="base", title="Calcite (mol)")

In [ ]:
# and for another realization
real = oe_pt.index[1]
plot_field("hk", real=real, log=True, title="H$_k$ (m/d)")
plot_field("poro", real=real, title="Porosity")
plot_field("calcite", real=real, title="Calcite (mol)")

In [ ]:
fig,ax = plt.subplots(1,1)
noise = pst.ies.noise
hobs = obs.loc[(~obs.obsnme.str.contains("gde")) & (obs.weight>0),:]
hobs.sort_values(by="obsval",inplace=True)
hvals = hobs.obsval.values
for real in oe_pt.index:
    vals = noise.loc[real,hobs.obsnme].values
    ax.plot(hvals,vals,"r-",marker=".",alpha=0.5)
    vals = oe_pr.loc[real,hobs.obsnme].values
    ax.plot(hvals, 
            vals,
            "0.5",
            # marker=".",
            alpha=0.5)
    vals = oe_pt.loc[real,hobs.obsnme].values
    ax.plot(hvals, 
            vals,
            "b",
            # marker=".",
            alpha=0.5,
            )

xlim = ax.get_xlim()
ylim = ax.get_ylim()
mn = min(xlim[0],ylim[0])
mx = max(xlim[1],ylim[1])
ax.plot([mn,mx], [mn,mx], "k--", lw=3)
ax.set_xlim(mn,mx)
ax.set_ylim(mn,mx)
ax.set_xlabel("Observed")
ax.set_ylabel("Simulated")
ax.set_title("Observed vs Simulated 1to1")

plt.tight_layout()

    